# Benchmarking a Novel L-Reduction Approach for Constrained QAOA

## 1. Introduction and Methodology
The Quantum Approximate Optimization Algorithm (QAOA) natively struggles with hard constraints. When solving the **Maximum Independent Set (MIS)** problem, standard approaches (like PennyLane's native `qaoa.max_independent_set`) map the constraints directly into the Hamiltonian, creating a complex energy landscape.

**Our Proposed Methodology:**
1. **L-Reduction to Max-2SAT:** We map the constrained MIS problem into an unconstrained Max-2SAT formulation. The goal of maximizing the independent set becomes a reward (weight 1.0) for picking a node, while the constraint (no adjacent nodes) becomes a penalty (weight 2.0) applied to the Max-2SAT clause $\neg u \lor \neg v$.
2. **Unconstrained QAOA:** We compile the Max-2SAT formula into an Ising Hamiltonian using standard mathematical encodings and solve it using QAOA. By removing hard constraints, the optimizer can navigate the landscape more smoothly.
3. **Classical Repair:** Because the Max-2SAT formulation is unconstrained, the quantum hardware may sample "invalid" bitstrings. We apply deterministic classical repair heuristics (Degree-based, Energy-based, Random, Clause-count) to project these invalid strings back into the valid solution space, keeping the best result.

This notebook evaluates if **QAOA + Max-2SAT + Classical Repair** outperforms the **Native Constrained QAOA** in terms of solution quality and execution time across varying graph sizes, maximum degrees, circuit depths ($p$), and shot counts.

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pennylane as qml
from pennylane import qaoa

from test_gen import graph_generator
from problems.mis_problem import MISProblem
from framework.qaoa_engine import QAOAEngine

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'legend.fontsize': 12,
    'figure.figsize': (10, 6)
})

In [ ]:
import pandas as pd
import time
import os
import networkx as nx

def run_comparative_benchmark(node_sizes, degrees, p_values, shots_values, graphs_per_config=3, data_dir="benchmark_assets"):
    graphs_path = os.path.join(data_dir, "graphs")
    os.makedirs(graphs_path, exist_ok=True)
    
    results_pl_unconstrained = []
    results_pl_constrained = []
    results_m2s_standard = []
    results_m2s_shifted = []
    results_m2s_normalized = []

    for n in node_sizes:
        for d in degrees:
            current_graphs = []
            all_exist = all([os.path.exists(os.path.join(graphs_path, f"graph_n{n}_d{d}_id{i}.gml")) for i in range(graphs_per_config)])
            
            if all_exist:
                for i in range(graphs_per_config):
                    G = nx.read_gml(os.path.join(graphs_path, f"graph_n{n}_d{d}_id{i}.gml"))
                    current_graphs.append(nx.convert_node_labels_to_integers(G))
            else:
                current_graphs = graph_generator.generate_graphs(num_graphs=graphs_per_config, num_nodes=n, max_degree=d)
                for i, G in enumerate(current_graphs):
                    nx.write_gml(G, os.path.join(graphs_path, f"graph_n{n}_d{d}_id{i}.gml"))

            for p in p_values:
                for s in shots_values:
                    for graph_idx, G in enumerate(current_graphs):
                        print(f"\n--- Testing: N={n}, D={d}, p={p}, S={s}, Graph={graph_idx+1} ---")
                        problem = MISProblem(G, penalty=2.0)
                        
                        # Get the 4 repair techniques from the problem instance
                        repair_contestants = problem.get_repair_strategies()

                        # ==========================================
                        # PENNYLANE NATIVE UNCONSTRAINED + TOURNAMENT
                        # ==========================================
                        print("[EXEC] Native Unconstrained (with Tournament)...")
                        start_un = time.time()
                        H_un, Hm_un = qaoa.max_independent_set(G, constrained=False)
                        params_un = QAOAEngine.optimize_qaoa(H_un, Hm_un, n, p=p)
                        samples_un = QAOAEngine.sample_qaoa(H_un, Hm_un, n, params_un, p=p, shots=s)
                        
                        best_un_score = -1
                        best_un_tech = "None"
                        for t_name, t_func in repair_contestants.items():
                            repaired = [t_func(bitstr) for bitstr in samples_un]
                            current_max = max([MISProblem.independent_set_size(r) for r in repaired])
                            if current_max > best_un_score:
                                best_un_score = current_max
                                best_un_tech = t_name
                        
                        results_pl_unconstrained.append({
                            'Nodes': n, 'Degree': d, 'p_Depth': p, 'Shots': s, 'Graph_ID': graph_idx,
                            'Best_Score': best_un_score, 'Best_Repair_Used': best_un_tech, 'Time_s': time.time() - start_un
                        })

                        # ==========================================
                        # PENNYLANE NATIVE CONSTRAINED + TOURNAMENT
                        # ==========================================
                        print("[EXEC] Native Constrained (with Tournament)...")
                        start_con = time.time()
                        H_con, Hm_con = qaoa.max_independent_set(G, constrained=True) 
                        params_con = QAOAEngine.optimize_qaoa(H_con, Hm_con, n, p=p)
                        samples_con = QAOAEngine.sample_qaoa(H_con, Hm_con, n, params_con, p=p, shots=s)
                        
                        best_con_score = -1
                        best_con_tech = "None"
                        for t_name, t_func in repair_contestants.items():
                            repaired = [t_func(bitstr) for bitstr in samples_con]
                            current_max = max([MISProblem.independent_set_size(r) for r in repaired])
                            if current_max > best_con_score:
                                best_con_score = current_max
                                best_con_tech = t_name

                        results_pl_constrained.append({
                            'Nodes': n, 'Degree': d, 'p_Depth': p, 'Shots': s, 'Graph_ID': graph_idx,
                            'Best_Score': best_con_score, 'Best_Repair_Used': best_con_tech, 'Time_s': time.time() - start_con
                        })

                        # ==========================================
                        # MAX-2SAT (Internal Tournament included)
                        # ==========================================
                        for style in ["standard", "shifted", "normalized"]:
                            print(f"[EXEC] Max-2SAT {style.capitalize()}...")
                            engine = QAOAEngine(p=p, shots=s, default_style=style)
                            start_m2s = time.time()
                            res = engine.run_experiment(problem) # Tournament happens inside here
                            
                            row = {
                                'Nodes': n, 'Degree': d, 'p_Depth': p, 'Shots': s, 'Graph_ID': graph_idx,
                                'Best_Score': res['best_score'],
                                'Best_Repair_Used': res['best_technique'],
                                'Time_s': time.time() - start_m2s
                            }
                            if style == "standard": results_m2s_standard.append(row)
                            elif style == "shifted": results_m2s_shifted.append(row)
                            else: results_m2s_normalized.append(row)

    # --- SAVE CSVs ---
    pd.DataFrame(results_pl_unconstrained).to_csv(os.path.join(data_dir, "results_pennylane_unconstrained.csv"), index=False)
    pd.DataFrame(results_pl_constrained).to_csv(os.path.join(data_dir, "results_pennylane_constrained.csv"), index=False)
    pd.DataFrame(results_m2s_standard).to_csv(os.path.join(data_dir, "results_max2sat_standard.csv"), index=False)
    pd.DataFrame(results_m2s_shifted).to_csv(os.path.join(data_dir, "results_max2sat_shifted.csv"), index=False)
    pd.DataFrame(results_m2s_normalized).to_csv(os.path.join(data_dir, "results_max2sat_normalized.csv"), index=False)
    

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

NODE_SIZES = [6, 8, 10, 12, 14, 16]
DEGREES = [3, 4, 5, 6]           
P_VALUES = [1, 2]       
SHOTS_VALUES = [500, 1000]    

print("Starting Comparative Benchmarks... Generating data for all encodings.")
# DECOMMENT THIS TO RE-RUN BENCHMARKS IF NEEDED. CURRENTLY COMMENTED OUT TO AVOID LONG EXECUTION TIMES DURING ANALYSIS.
# In practice, you would run this cell once to generate the CSVs, and then comment it out for all subsequent analysis and plotting to save time.
# In the directory "benchmark_assets", the generated CSV files will be saved, which can then be loaded for analysis and plotting without needing to re-run the benchmarks.
# Running with the next function call commented, will produce the plots you see in the paper, based on the previously generated CSV files.
# run_comparative_benchmark(
#     node_sizes=NODE_SIZES, 
#     degrees=DEGREES, 
#     p_values=P_VALUES, 
#     shots_values=SHOTS_VALUES, 
#     graphs_per_config=10
# )
print("\nBenchmark completed with unified Repair Tournament logic.")

ASSETS_DIR = "benchmark_assets"
file_map = {
    "results_pennylane_unconstrained.csv": "Native Unconstrained",
    "results_pennylane_constrained.csv": "Native Constrained",
    "results_max2sat_standard.csv": "M2S Standard",
    "results_max2sat_shifted.csv": "M2S Shifted",
    "results_max2sat_normalized.csv": "M2S Normalized"
}

all_dfs = []

for filename, label in file_map.items():
    path = os.path.join(ASSETS_DIR, filename)
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['Method'] = label
        all_dfs.append(df)

if not all_dfs:
    raise FileNotFoundError(f"No benchmark CSVs found in {ASSETS_DIR}")

df_all = pd.concat(all_dfs, ignore_index=True)

m2s_labels = ["M2S Standard", "M2S Shifted", "M2S Normalized"]
m2s_performance = df_all[df_all['Method'].isin(m2s_labels)].groupby('Method')['Best_Score'].mean()
best_m2s = m2s_performance.idxmax()

print(f"Best Max-2SAT Variant Detected: {best_m2s}")

baselines = ["Native Unconstrained", "Native Constrained"]
df_final = df_all[df_all['Method'].isin(baselines + [best_m2s])].copy()

df_final.loc[df_final['Method'] == best_m2s, 'Method'] = f"{best_m2s} (Proposed)"

summary = df_final.groupby(['Nodes', 'Method'])['Best_Score'].mean().unstack()
summary['Winner'] = summary.idxmax(axis=1)
print("\n--- Summary Table (Mean MIS Size) ---")
display(summary) 

sns.set_theme(style="whitegrid", context="talk")
fig, ax = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Solution Quality
sns.lineplot(data=df_final, x='Nodes', y='Best_Score', hue='Method', 
             marker='o', markersize=10, errorbar='sd', ax=ax[0], linewidth=3)
ax[0].set_title("Solution Quality Scaling", fontsize=16, weight='bold')
ax[0].set_ylabel("Mean MIS Size")
ax[0].set_xlabel("Problem Size (N)")

# Plot 2: Runtime Efficiency
sns.lineplot(data=df_final, x='Nodes', y='Time_s', hue='Method', 
             marker='s', markersize=10, errorbar='sd', ax=ax[1], linewidth=3)
ax[1].set_yscale('log')
ax[1].set_title("Runtime Scalability (Log Scale)", fontsize=16, weight='bold')
ax[1].set_ylabel("Execution Time (seconds)")
ax[1].set_xlabel("Problem Size (N)")

plt.tight_layout()
plt.show()